# Boston Housing Data Analysis
## Statistics for Data Science with Python - Final Project

**Project Title:** Statistical Analysis of Boston Housing Dataset  
**Course:** Statistics for Data Science with Python (IBM / Coursera)  
**Author:** [Your Name]  
**Date:** October 2026

---

### Project Overview
In this project, we act as Data Scientists at a housing agency. We analyze the Boston Housing dataset to generate insights about median house values and their relationships with various socio-economic and environmental factors using descriptive statistics, hypothesis testing, correlation, and regression analysis.

## Task 1: Become familiar with the dataset

The following describes the dataset variables:

- **CRIM** - per capita crime rate by town
- **ZN** - proportion of residential land zoned for lots over 25,000 sq.ft.
- **INDUS** - proportion of non-retail business acres per town
- **CHAS** - Charles River dummy variable (1 if tract bounds river; 0 otherwise)
- **NOX** - nitric oxides concentration (parts per 10 million)
- **RM** - average number of rooms per dwelling
- **AGE** - proportion of owner-occupied units built prior to 1940
- **DIS** - weighted distances to five Boston employment centres
- **RAD** - index of accessibility to radial highways
- **TAX** - full-value property-tax rate per $10,000
- **PTRATIO** - pupil-teacher ratio by town
- **LSTAT** - % lower status of the population
- **MEDV** - Median value of owner-occupied homes in $1000's

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

%matplotlib inline
sns.set(style="whitegrid")

In [ ]:
# Load the dataset
boston_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ST0151EN-SkillsNetwork/labs/boston_housing.csv'
boston_df = pd.read_csv(boston_url)

print("Dataset shape:", boston_df.shape)
boston_df.head()

In [ ]:
boston_df.info()
boston_df.describe()

## Task 2: Generate Descriptive Statistics and Visualizations

### 2.1 Boxplot for Median value of owner-occupied homes (MEDV)

In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(y=boston_df['MEDV'])
plt.title('Boxplot of Median Value of Owner-Occupied Homes (MEDV)')
plt.ylabel("Median value in $1000's")
plt.show()

print("Median of MEDV:", boston_df['MEDV'].median())
print("There are several outliers above the upper whisker.")

### 2.2 Bar plot for the Charles River variable (CHAS)

In [ ]:
plt.figure(figsize=(7,5))
sns.countplot(x=boston_df['CHAS'])
plt.title('Bar Plot of Charles River Dummy Variable (CHAS)')
plt.xlabel('Bounds Charles River (1 = Yes, 0 = No)')
plt.ylabel('Count')
plt.show()

print(boston_df['CHAS'].value_counts())

### 2.3 Boxplot for MEDV vs AGE (discretized into 3 groups)

In [ ]:
# Discretize AGE into three groups
boston_df.loc[(boston_df['AGE'] <= 35), 'AGE_Group'] = '35 years and younger'
boston_df.loc[(boston_df['AGE'] > 35) & (boston_df['AGE'] < 70), 'AGE_Group'] = 'Between 35 and 70 years'
boston_df.loc[(boston_df['AGE'] >= 70), 'AGE_Group'] = '70 years and older'

plt.figure(figsize=(10,6))
sns.boxplot(x='AGE_Group', y='MEDV', data=boston_df,
            order=['35 years and younger', 'Between 35 and 70 years', '70 years and older'])
plt.title('Boxplot of MEDV vs AGE Groups')
plt.xlabel('Age Group of Owner-Occupied Units')
plt.ylabel("Median value in $1000's")
plt.show()

### 2.4 Scatter plot of Nitric Oxide Concentrations (NOX) vs Non-retail business acres (INDUS)

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(x='INDUS', y='NOX', data=boston_df)
plt.title('Scatter Plot: NOX vs INDUS')
plt.xlabel('Proportion of Non-Retail Business Acres (INDUS)')
plt.ylabel('Nitric Oxides Concentration (NOX)')
plt.show()

print("There appears to be a positive relationship: higher industrial proportion is associated with higher NOX levels.")

### 2.5 Histogram for Pupil-Teacher Ratio (PTRATIO)

In [ ]:
plt.figure(figsize=(8,6))
sns.histplot(boston_df['PTRATIO'], bins=15, kde=True)
plt.title('Histogram of Pupil-Teacher Ratio (PTRATIO)')
plt.xlabel('Pupil-Teacher Ratio by Town')
plt.ylabel('Frequency')
plt.show()

print("The distribution of PTRATIO is slightly left-skewed with a concentration around 20.")

## Task 3: Use the appropriate tests to answer the questions provided

### 3.1 Is there a significant difference in median value of houses bounded by the Charles River or not? (T-test for independent samples)

**Hypothesis:**
- H₀: µ₁ = µ₂ (No difference in mean MEDV between houses bounded by Charles River and those that are not)
- H₁: µ₁ ≠ µ₂ (There is a difference)

In [ ]:
chas_1 = boston_df[boston_df['CHAS'] == 1]['MEDV']
chas_0 = boston_df[boston_df['CHAS'] == 0]['MEDV']

# Levene test for equality of variances
levene_stat, levene_p = scipy.stats.levene(chas_1, chas_0, center='mean')
print(f"Levene test p-value: {levene_p:.4f}")

# T-test
t_stat, p_value = scipy.stats.ttest_ind(chas_1, chas_0, equal_var=(levene_p > 0.05))
print(f"T-test statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("\nConclusion: Reject H₀. There is a significant difference in median house values for houses bounded by the Charles River.")
else:
    print("\nConclusion: Fail to reject H₀. No significant difference found.")

### 3.2 Is there a difference in Median values of houses (MEDV) for each proportion of owner occupied units built prior to 1940 (AGE)? (ANOVA)

**Hypothesis:**
- H₀: µ₁ = µ₂ = µ₃ (No difference in mean MEDV across the three AGE groups)
- H₁: At least one mean is different

In [ ]:
group1 = boston_df[boston_df['AGE_Group'] == '35 years and younger']['MEDV']
group2 = boston_df[boston_df['AGE_Group'] == 'Between 35 and 70 years']['MEDV']
group3 = boston_df[boston_df['AGE_Group'] == '70 years and older']['MEDV']

f_stat, p_value = scipy.stats.f_oneway(group1, group2, group3)
print(f"ANOVA F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("\nConclusion: Reject H₀. There is a significant difference in MEDV across AGE groups.")
else:
    print("\nConclusion: Fail to reject H₀. No significant difference found.")

### 3.3 Can we conclude that there is no relationship between Nitric oxide concentrations and proportion of non-retail business acres per town? (Pearson Correlation)

**Hypothesis:**
- H₀: ρ = 0 (No correlation between NOX and INDUS)
- H₁: ρ ≠ 0 (There is a correlation)

In [ ]:
corr_coef, p_value = scipy.stats.pearsonr(boston_df['NOX'], boston_df['INDUS'])
print(f"Pearson Correlation Coefficient: {corr_coef:.4f}")
print(f"P-value: {p_value:.4e}")

if p_value < 0.05:
    print("\nConclusion: Reject H₀. There is a statistically significant correlation between NOX and INDUS.")
else:
    print("\nConclusion: Fail to reject H₀. No significant correlation found.")

### 3.4 What is the impact of an additional weighted distance to the five Boston employment centres on the median value of owner occupied homes? (Regression Analysis)

In [ ]:
X = boston_df['DIS']
y = boston_df['MEDV']

X = sm.add_constant(X)  # add intercept

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
print("\nInterpretation:")
print(f"Coefficient of DIS: {model.params['DIS']:.4f}")
print("An additional unit of weighted distance to employment centres is associated with an increase of approximately $1,092 in the median home value (holding other factors constant in this simple model).")
print(f"R-squared: {model.rsquared:.4f} → The model explains only about {model.rsquared*100:.1f}% of the variance in MEDV.")

## Conclusion

In this project we performed a comprehensive statistical analysis of the Boston Housing dataset:

1. **Descriptive Statistics & Visualizations** revealed the distribution of median home values, the imbalance in Charles River proximity, the relationship between house age and value, the positive association between industrialization and pollution, and the distribution of pupil-teacher ratios.

2. **Hypothesis Testing** showed:
   - Significant difference in MEDV for houses near the Charles River.
   - Significant difference in MEDV across different AGE groups.
   - Strong positive correlation between NOX and INDUS.

3. **Regression Analysis** indicated that greater distance to employment centres is positively associated with higher median home values in this simple model, although the explanatory power is limited.

These insights can help a housing agency better understand the factors influencing property values in the Boston area.